<a href="https://colab.research.google.com/github/AlejandroMario-creator/Bot_Scraping/blob/main/scraping_basico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup


def extraer_pagina(url_actual):
    """Procesa una sola página de Books to Scrape.

    Devuelve:
      - lista de libros encontrados en esa página
      - la URL de la página siguiente (o None si es la última)
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    try:
        respuesta = requests.get(url_actual, headers=headers, timeout=10)
        respuesta.raise_for_status()
    except requests.RequestException as e:
        print(f"Error al conectar a {url_actual}: {e}")
        return [], None

    soup = BeautifulSoup(respuesta.text, "html.parser")

    # 1. Extraer libros de la página actual
    libros_pagina = []
    articulos = soup.find_all("article", class_="product_pod")

    for art in articulos:
        titulo = art.h3.a["title"]
        precio = art.find("p", class_="price_color").text.strip()
        disponibilidad = art.find(
            "p", class_="instock availability"
        ).text.strip()

        # Enlace individual
        rel_link = art.h3.a["href"]
        link_completo = urljoin(url_actual, rel_link)

        libros_pagina.append(
            {
                "titulo": titulo,
                "precio": precio,
                "disponibilidad": disponibilidad,
                "url": link_completo,
            }
        )

    # 2. Buscar si existe botón de "Página Siguiente" (<li class="next"><a href="...">)
    boton_siguiente = soup.find("li", class_="next")
    url_siguiente = None

    if boton_siguiente and boton_siguiente.find("a"):
        href_siguiente = boton_siguiente.find("a")["href"]
        url_siguiente = urljoin(url_actual, href_siguiente)

    return libros_pagina, url_siguiente

In [ ]:
import time
import pandas as pd

def ejecutar_bot(url_inicial, limite_pag = None):
  todos_libros = []
  url_actual = url_inicial
  contador_pag = 0

  print("Iniciando bot....")

  while url_actual:
    contador_pag += 1
    print(f"Procesando pagina {contador_pag}: {url_actual}")

    libros_pagina, url_actual = extraer_pagina(url_actual)
    todos_libros.extend(libros_pagina)

    time.sleep(0.5)

    if limite_pag and contador_pag >= limite_pag:
      print(f"Limite de paginas alcanzado {limite_pag}")
      break

  print(f"Bot finalizado. Se extrajeron {len(todos_libros)}")

  df = pd.DataFrame(todos_libros)
  df.to_csv("books_to_scrape.csv", index=False, encoding="utf-8-sig")
  print("Datos guardados exitosamente")
  return df

In [ ]:
ejecutar_bot(
    "https://books.toscrape.com/", limite_pag=3
)

Iniciando bot....
Procesando pagina 1: https://books.toscrape.com/
Procesando pagina 2: https://books.toscrape.com/catalogue/page-2.html
Procesando pagina 3: https://books.toscrape.com/catalogue/page-3.html
Limite de paginas alcanzado 3
Bot finalizado. Se extrajeron 60
Datos guardados exitosamente


,titulo,precio,disponibilidad,url
0,A Light in the Attic,Â£51.77,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,Â£22.65,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,Â£52.15,In stock,https://books.toscrape.com/catalogue/the-black...
